In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [4]:
v1.dot(dv)

np.float32(0.32332397)

In [5]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [6]:
v2.dot(dv)

np.float32(0.019730574)

In [7]:
from ingest import load_faq_data

documents = load_faq_data()

In [8]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [9]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1380

In [10]:
vectors[10].shape

(384,)

In [11]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [12]:
import numpy as np
X = np.array(vectors)

In [13]:
scores = X.dot(v1)

In [14]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(860), np.float32(0.762941))

In [15]:
top5 = np.argsort(-scores)[:5]

In [16]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

In [17]:
top5

array([ 860,  473,   29, 1262,  865])

In [18]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [19]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [20]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [21]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project s

In [22]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
anthropic_client = Anthropic()

In [23]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [24]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=anthropic_client,
)

In [25]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

("# Yes, you can still sign up!\n\nAccording to the course information, **yes, you can still join the program even if you just discovered it**.\n\nHowever, there is one important condition: **if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.**\n\nA few additional things to know:\n- **Homework is not mandatory** to get a certificate — you need to pass the Capstone project to receive one\n- **Homework is still recommended** as it helps reinforce concepts and earns you points on the leaderboard\n- **Submission forms remain open** even after listed deadlines pass, so you can still submit as long as the form itself hasn't been closed\n\nI'd recommend getting started as soon as possible to ensure you have time to complete the coursework and submit your capstone project!",
 Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, i

In [26]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [27]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=anthropic_client,
)

In [28]:
vector_assistant.rag("the program has already begun, can I still sign up?")

('Based on the provided context, **yes, you can still sign up!**\n\nAccording to the course information:\n\n> "Yes, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions."\n\nSo you can join the program even though it has already begun. However, there\'s an important condition: if you want to receive a certificate, make sure to submit your project while submissions are still being accepted.\n\nYou can also start learning and submitting homework as long as the submission forms remain open.',
 Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=419, output_tokens=115, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [29]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [30]:
vs_index.fit(vectors, documents)